[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JulesMalin/isba2411-nlp/blob/main/Week%203/ex_gemini_demo.ipynb)

# ISBA 2411 - Live Demo: Coding with Gemini in Colab

**Goal (15 min).** Let the AI write the code, and practice the part that stays human: knowing whether the code is actually right.

We will build a movie-review sentiment classifier with TF-IDF and logistic regression, with Gemini doing the typing and you doing the judging.

**Setup before class:** open the Gemini side panel (the spark icon, upper right). Run the cell below once so the runtime is warm.

In [ ]:
# Warm up the runtime and confirm packages are present.
!pip -q install datasets scikit-learn 2>/dev/null
import sklearn, datasets
print('scikit-learn', sklearn.__version__)
print('datasets', datasets.__version__)

---
## 1. Let the AI teach you to code  `0:02 - 0:06`

Four quick features, no model yet.

1. **Ask in plain English.** In the Gemini panel type: *What does TF-IDF do, in one paragraph?*
2. **Inline autocomplete.** In the next cell, start typing and press **Tab** to accept the gray suggestion.
3. **Explain error.** Run the error cell, then click **Explain error** under the traceback.
4. **Explain code.** Right-click any cell and choose *Explain code* to read unfamiliar code in plain English.

In [ ]:
# Autocomplete demo: type the first few letters and press Tab to let Gemini finish the line.
# Try:  import pandas as



In [ ]:
# Explain-error demo: run this on purpose, then click 'Explain error'.
prnt('hello NLP')

---
## 2. Build the model with Gemini  `0:06 - 0:14`

Paste this into the Gemini panel, then drop its code into the empty cell below and run it:

> Write Colab code to train a sentiment classifier on the rotten_tomatoes dataset using TF-IDF and logistic regression with scikit-learn. Print accuracy and a classification report.

Run whatever it gives you first. Then compare against the reference cell further down and steer Gemini toward the gaps.

In [ ]:
# >>> Let Gemini generate the model here, then run it. <<<



### Reference / fallback (instructor only)

Known-good version. Use it if Gemini drifts or if you want a target to steer toward. Lands around 76-78% accuracy, a clean teaching baseline.

In [ ]:
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

ds = load_dataset('rotten_tomatoes')
Xtr_txt, ytr = ds['train']['text'], ds['train']['label']
Xte_txt, yte = ds['test']['text'],  ds['test']['label']

vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, stop_words='english')
Xtr = vec.fit_transform(Xtr_txt)
Xte = vec.transform(Xte_txt)

clf = LogisticRegression(max_iter=1000)
clf.fit(Xtr, ytr)

pred = clf.predict(Xte)
print('accuracy:', round(accuracy_score(yte, pred), 4))
print(classification_report(yte, pred, target_names=['negative', 'positive']))

---
## 3. Verify and improve  `0:14 - 0:17`

This is the point of the segment. Gemini's first draft usually misses something. Look for:

- No n-grams (default `ngram_range=(1, 1)`), so it ignores word pairs like *not good*.
- No `min_df`, so rare noise words inflate the feature space.
- Evaluating on the training set instead of the held-out test split.

Name the weakness out loud, ask Gemini to fix that one thing, and watch the number move. Then sanity-check a sentence yourself.

In [ ]:
# Quick human check. Match vec/clf to whatever names Gemini used.
samples = [
    'a dull, lifeless mess with no redeeming moments',
    'a sharp, funny film that earns its ending',
]
labels = {0: 'negative', 1: 'positive'}
for s, p in zip(samples, clf.predict(vec.transform(samples))):
    print(f'{labels[p]:>8}  <-  {s}')

**Improvement prompt for Gemini:**

> Add bigrams and a minimum document frequency to the TF-IDF vectorizer, and show the accuracy before and after.

**The lesson:** the code arrived in seconds. The judgment about whether it was any good was yours.

---
## Fallback: dataset will not download

If the room wifi blocks the dataset, upload `reviews.csv` (columns: `text`, `label`) to the Colab file panel and run the cell below instead of `load_dataset`. Everything downstream stays the same.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv('reviews.csv')
Xtr_txt, Xte_txt, ytr, yte = train_test_split(
    df['text'].tolist(), df['label'].tolist(), test_size=0.2, random_state=42
)
print('train:', len(Xtr_txt), '  test:', len(Xte_txt))